In [1]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.3 MB/s eta 0:00:00


In [2]:
import yaml, shutil

# Just copy the yaml file only — not the whole dataset
shutil.copy(
    '/kaggle/input/datasets/vishweshvishwakarma/geo-dataset/yolo_split/dataset.yaml',
    '/kaggle/working/dataset.yaml'
)

# Fix the yaml to point back to input folder
yaml_path = '/kaggle/working/dataset.yaml'
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['path']  = '/kaggle/input/datasets/vishweshvishwakarma/geo-dataset/yolo_split/'
data['train'] = 'train/images'
data['val']   = 'val/images'
data['test']  = 'test/images'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)
print("Done!")
print(open(yaml_path).read())

Done!
names:
  0: BuiltUp_Roof1
  1: BuiltUp_Roof2
  2: BuiltUp_Roof3
  3: BuiltUp_Roof4
  4: Road_1
  5: Road_3
  6: Road_4
  7: Road_5
  8: Road_6
  9: Utility
  10: Water
  11: Bridge
  12: Railway
nc: 13
path: /kaggle/input/datasets/vishweshvishwakarma/geo-dataset/yolo_split/
test: test/images
train: train/images
val: val/images



In [3]:
from ultralytics import YOLO

# 1. Load the weights that reached 91% precision
model = YOLO('/kaggle/input/models/vishweshvishwakarma/yolo26-custom-best/pytorch/default/1/best.pt')

# 2. Execute the high-precision 10-epoch sprint
results = model.train(
    data='/kaggle/working/dataset.yaml',
    task='segment',
    epochs=10,               # Short, high-intensity polish
    imgsz=768,               
    batch=16,                # Maximize GPU utilization
    device=[0, 1],           
    optimizer='AdamW',
    lr0=1e-5,                # Start very low to prevent "weight shock"
    lrf=0.01,                
    warmup_epochs=0,         # Start immediate tuning
    cos_lr=True,             # Decay to almost zero
    freeze=10,               # 🧊 Lock the backbone (Layers 0-9)
    mosaic=0.0,              # ❌ No mosaic (focus on real-world tile geometry)
    close_mosaic=0,          # Already disabled
    overlap_mask=True,       
    project='/kaggle/working/yolo26_m_seg_project',
    name='run_v4',
    exist_ok=True,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.26 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=

In [4]:
import os
import shutil
import glob

# 1. Automatically find the newest 'best.pt' wherever YOLO put it
weight_files = glob.glob('/kaggle/working/yolo26_m_seg_project/**/best.pt', recursive=True)

if weight_files:
    # Sort by creation time to get the absolute newest one from this specific run
    latest_weight = max(weight_files, key=os.path.getctime)
    print(f"Found Diamond Weights at: {latest_weight}")
    
    # Copy to the top level for easy individual download
    shutil.copy(latest_weight, '/kaggle/working/diamond_best.pt')
    
    # 2. ZIP the entire project folder so you get the curves and logs too
    shutil.make_archive('/kaggle/working/final_diamond_results', 'zip', '/kaggle/working/yolo26_m_seg_project')
    print("✅ All files found, copied, and zipped! You are safe to close the tab.")
else:
    print("⚠️ Warning: best.pt not found. Check the /kaggle/working/ folder manually.")

Found Diamond Weights at: /kaggle/working/yolo26_m_seg_project/run_v4/weights/best.pt
✅ All files found, copied, and zipped! You are safe to close the tab.


In [5]:
# import shutil

# shutil.make_archive(
#     '/kaggle/working/yolo_run_m_3',  # output zip name
#     'zip',
#     '/kaggle/working/yolo26_m_seg_project/run_v3'   # folder to zip
# )
# print("Done!")